# Disease Prediction based on Symtoms

In [1]:
#Importing Libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
# Libraries to clean data
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
# Libraries to perform feature selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
# Libraries to train models
from sklearn.model_selection import GridSearchCV
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
# Libraries to evaluate performance
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score, precision_score

# Cleaning the Data

In [2]:
# Reading the Training.csv file
df_train = pd.read_csv("data/Training.csv")

In [3]:
# Reading the Training.csv file
df_test = pd.read_csv("data/Testing.csv")

In [4]:
df_test = df_test.tail(1)

In [5]:
df_test.columns = df_test.columns.str.replace('_', ' ')

In [6]:
# Splitting data into features and target
X2 = df_test.drop('prognosis', axis=1)
y2 = df_test[["prognosis"]]
y2 = np.ravel(y2)

In [7]:
# Dropping unnecessary columns
df_train = df_train.drop('Unnamed: 133', axis=1)
df_train.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [8]:
# List of columns names (i.e.symptoms)
df_train.columns.tolist()

['itching',
 'skin_rash',
 'nodal_skin_eruptions',
 'continuous_sneezing',
 'shivering',
 'chills',
 'joint_pain',
 'stomach_pain',
 'acidity',
 'ulcers_on_tongue',
 'muscle_wasting',
 'vomiting',
 'burning_micturition',
 'spotting_ urination',
 'fatigue',
 'weight_gain',
 'anxiety',
 'cold_hands_and_feets',
 'mood_swings',
 'weight_loss',
 'restlessness',
 'lethargy',
 'patches_in_throat',
 'irregular_sugar_level',
 'cough',
 'high_fever',
 'sunken_eyes',
 'breathlessness',
 'sweating',
 'dehydration',
 'indigestion',
 'headache',
 'yellowish_skin',
 'dark_urine',
 'nausea',
 'loss_of_appetite',
 'pain_behind_the_eyes',
 'back_pain',
 'constipation',
 'abdominal_pain',
 'diarrhoea',
 'mild_fever',
 'yellow_urine',
 'yellowing_of_eyes',
 'acute_liver_failure',
 'fluid_overload',
 'swelling_of_stomach',
 'swelled_lymph_nodes',
 'malaise',
 'blurred_and_distorted_vision',
 'phlegm',
 'throat_irritation',
 'redness_of_eyes',
 'sinus_pressure',
 'runny_nose',
 'congestion',
 'chest_pain',


In [9]:
# Check null columns 
null_cols = [col for col in df_train.columns if df_train[col].isnull().sum() > 0]
null_cols

[]

In [10]:
df_train.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [11]:
df_train.columns = df_train.columns.str.replace('_', ' ')

In [12]:
# List of prognosis
df_train['prognosis'].value_counts().sort_index()

(vertigo) Paroymsal  Positional Vertigo    120
AIDS                                       120
Acne                                       120
Alcoholic hepatitis                        120
Allergy                                    120
Arthritis                                  120
Bronchial Asthma                           120
Cervical spondylosis                       120
Chicken pox                                120
Chronic cholestasis                        120
Common Cold                                120
Dengue                                     120
Diabetes                                   120
Dimorphic hemmorhoids(piles)               120
Drug Reaction                              120
Fungal infection                           120
GERD                                       120
Gastroenteritis                            120
Heart attack                               120
Hepatitis B                                120
Hepatitis C                                120
Hepatitis D  

In [13]:
# Splitting data into features and target
X = df_train.drop('prognosis', axis=1)
y = df_train[["prognosis"]]
y = np.ravel(y)

In [14]:
# Split data into training and testing sets
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, train_size = 0.50, random_state=42)

In [15]:
X_train_temp.shape, X_test.shape, y_train_temp.shape, y_test.shape

((2460, 132), (2460, 132), (2460,), (2460,))

In [16]:
# Split training set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, train_size = 0.70, random_state=42)

In [17]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((1722, 132), (738, 132), (1722,), (738,))

# Performing Feature Selection

**To select the top 95 features, we used RFE with Random Forest as an estimator**

In [18]:
rfc = RandomForestClassifier()
n_features = 95
rfe = RFE(estimator=rfc, n_features_to_select=n_features)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)
X_val_rfe = rfe.transform(X_val)

In [19]:
X2_rfe = rfe.transform(X2)

In [20]:
# Get the names of the selected features
selected_feature_names = df_train.columns[:-1][rfe.support_]
selected_feature_names = list(selected_feature_names)
print("Selected features:", selected_feature_names)

Selected features: ['itching', 'skin rash', 'nodal skin eruptions', 'continuous sneezing', 'shivering', 'chills', 'joint pain', 'stomach pain', 'acidity', 'ulcers on tongue', 'muscle wasting', 'vomiting', 'burning micturition', 'spotting  urination', 'fatigue', 'weight loss', 'lethargy', 'patches in throat', 'cough', 'high fever', 'breathlessness', 'sweating', 'indigestion', 'headache', 'yellowish skin', 'dark urine', 'nausea', 'loss of appetite', 'back pain', 'constipation', 'abdominal pain', 'diarrhoea', 'mild fever', 'yellowing of eyes', 'swelled lymph nodes', 'malaise', 'blurred and distorted vision', 'phlegm', 'sinus pressure', 'chest pain', 'fast heart rate', 'irritation in anus', 'neck pain', 'dizziness', 'cramps', 'swollen legs', 'enlarged thyroid', 'swollen extremeties', 'excessive hunger', 'extra marital contacts', 'slurred speech', 'knee pain', 'hip joint pain', 'muscle weakness', 'stiff neck', 'swelling joints', 'movement stiffness', 'spinning movements', 'loss of balance',

# Train the Models using Selected Features

**To build the precision of the model, we utilized three distinctive algorithms which are as per the following**
* Decision Tree algorithm
* Random Forest algorithm
* KNearestNeighbour algorithm
* Naive Bayes algorithm
* Support Vector Machines algorithm
* Gradient Boosting algorithm

## Decision Tree Algorithm

In [21]:
dtc = tree.DecisionTreeClassifier() 
dtc = dtc.fit(X_train_rfe,y_train)

y_pred=dtc.predict(X_train_rfe)
print("Decision Tree")
print("Accuracy of training set")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred,normalize=False))

Decision Tree
Accuracy of training set
Accuracy score: 1.0
Number of correct predictions: 1722


In [22]:
y_pred=dtc.predict(X_val_rfe)
print("Decision Tree")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Decision Tree
Accuracy of validation set
Accuracy score: 0.9905149051490515
Number of correct predictions: 731


In [23]:
dtc2 = tree.DecisionTreeClassifier() 
dtc2 = dtc.fit(X_train_rfe,y_train)

y_pred=dtc2.predict(X_test_rfe)
print("Decision Tree")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Decision Tree
Accuracy of testing set
Accuracy score: 0.9752032520325203
Number of correct predictions: 2399


In [24]:
y_pred=dtc2.predict(X2_rfe)
print("Decision Tree")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Decision Tree
Accuracy of random sample 42
Accuracy score: 0.0
Number of correct predictions: 0


,Actual,Predicted
0,Dengue,Fungal infection


## Random Forest Algorithm

In [25]:
rfc = RandomForestClassifier()
rfc = rfc.fit(X_train_rfe,y_train)

# calculating accuracy 
y_pred=rfc.predict(X_train_rfe)
print("Random Forest")
print("Accuracy of training set")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred,normalize=False))

Random Forest
Accuracy of training set
Accuracy score: 1.0
Number of correct predictions: 1722


In [26]:
# calculating accuracy 
y_pred=rfc.predict(X_val_rfe)
print("Random Forest")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Random Forest
Accuracy of validation set
Accuracy score: 1.0
Number of correct predictions: 738


In [27]:
rfc = RandomForestClassifier()
rfc = rfc.fit(X_train_rfe,y_train)

# calculating accuracy 
y_pred=rfc.predict(X_test_rfe)
print("Random Forest")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Random Forest
Accuracy of testing set
Accuracy score: 1.0
Number of correct predictions: 2460


In [28]:
y_pred=rfc.predict(X2_rfe)
print("Random Forest")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Random Forest
Accuracy of random sample 42
Accuracy score: 0.0
Number of correct predictions: 0


,Actual,Predicted
0,Impetigo,Fungal infection


## Gradient Boosting

In [29]:
gbc = GradientBoostingClassifier()
gbc=gbc.fit(X_train_rfe,y_train)

y_pred=gbc.predict(X_train_rfe)
print("Gradient Boosting")
print("Accuracy of training set")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred,normalize=False))

Gradient Boosting
Accuracy of training set
Accuracy score: 1.0
Number of correct predictions: 1722


In [30]:
y_pred=gbc.predict(X_val_rfe)
print("Gradient Boosting")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Gradient Boosting
Accuracy of validation set
Accuracy score: 0.994579945799458
Number of correct predictions: 734


In [31]:
y_pred=gbc.predict(X_test_rfe)
print("Gradient Boosting")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Gradient Boosting
Accuracy of testing set
Accuracy score: 0.9943089430894309
Number of correct predictions: 2446


In [32]:
y_pred=gbc.predict(X2_rfe)
print("Gradient Boosting")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Gradient Boosting
Accuracy of random sample 42
Accuracy score: 0.0
Number of correct predictions: 0


,Actual,Predicted
0,Impetigo,Fungal infection


## Support Vector Machine

In [33]:
svc = SVC()
svc=svc.fit(X_train_rfe,y_train)

y_pred=svc.predict(X_train_rfe)
print("Support Vector Machine")
print("Accuracy of training set")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred,normalize=False))

Support Vector Machine
Accuracy of training set
Accuracy score: 1.0
Number of correct predictions: 1722


In [34]:
y_pred=svc.predict(X_val_rfe)
print("Support Vector Machine")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Support Vector Machine
Accuracy of validation set
Accuracy score: 1.0
Number of correct predictions: 738


In [35]:
y_pred=svc.predict(X_test_rfe)
print("Support Vector Machine")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Support Vector Machine
Accuracy of testing set
Accuracy score: 1.0
Number of correct predictions: 2460


In [36]:
y_pred=svc.predict(X2_rfe)
print("Support Vector Machine")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Support Vector Machine
Accuracy of random sample 42
Accuracy score: 1.0
Number of correct predictions: 1


,Actual,Predicted
0,Fungal infection,Fungal infection


## Naive bayes

In [37]:
nb = GaussianNB()
nb.fit(X_train_rfe, y_train)

y_pred = nb.predict(X_train_rfe)
print("Naive Bayes")
print("Accuracy on the training set:")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred, normalize=False))

Naive Bayes
Accuracy on the training set:
Accuracy score: 1.0
Number of correct predictions: 1722


In [38]:
y_pred=nb.predict(X_val_rfe)
print("Naive Bayes")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Naive Bayes
Accuracy of validation set
Accuracy score: 0.997289972899729
Number of correct predictions: 736


In [39]:
y_pred=nb.predict(X_test_rfe)
print("Naive Bayes")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Naive Bayes
Accuracy of testing set
Accuracy score: 0.9983739837398374
Number of correct predictions: 2456


In [40]:
y_pred=nb.predict(X2_rfe)
print("Naive Bayes")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Naive Bayes
Accuracy of random sample 42
Accuracy score: 1.0
Number of correct predictions: 1


,Actual,Predicted
0,Fungal infection,Fungal infection


## KNN Neighbors

In [41]:
knn = KNeighborsClassifier()
knn.fit(X_train_rfe, y_train)

y_pred = knn.predict(X_train_rfe)
print("K-Nearest Neighbors")
print("Accuracy on the training set:")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred, normalize=False))

K-Nearest Neighbors
Accuracy on the training set:
Accuracy score: 1.0
Number of correct predictions: 1722


In [42]:
y_pred=knn.predict(X_val_rfe)
print("K-Nearest Neighbors")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

K-Nearest Neighbors
Accuracy of validation set
Accuracy score: 1.0
Number of correct predictions: 738


In [43]:
y_pred=knn.predict(X_test_rfe)
print("K-Nearest Neighbors")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

K-Nearest Neighbors
Accuracy of testing set
Accuracy score: 1.0
Number of correct predictions: 2460


In [44]:
y_pred=knn.predict(X2_rfe)
print("K-Nearest Neighbors")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

K-Nearest Neighbors
Accuracy of random sample 42
Accuracy score: 1.0
Number of correct predictions: 1


,Actual,Predicted
0,Fungal infection,Fungal infection


## Logistic Regression

In [45]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train_rfe, y_train)

y_pred = logreg.predict(X_train_rfe)
print("Logistic Regression")
print("Accuracy on the training set:")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred, normalize=False))

Logistic Regression
Accuracy on the training set:
Accuracy score: 1.0
Number of correct predictions: 1722


In [46]:
y_pred=logreg.predict(X_val_rfe)
print("Logistic Regression")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Logistic Regression
Accuracy of validation set
Accuracy score: 1.0
Number of correct predictions: 738


In [47]:
y_pred=logreg.predict(X_test_rfe)
print("Logistic Regression")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Logistic Regression
Accuracy of testing set
Accuracy score: 1.0
Number of correct predictions: 2460


In [48]:
y_pred=logreg.predict(X2_rfe)
print("Logistic Regression")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Logistic Regression
Accuracy of random sample 42
Accuracy score: 1.0
Number of correct predictions: 1


,Actual,Predicted
0,Fungal infection,Fungal infection


## Linear Regression

In [49]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_rfe, y_train)

y_pred = svm.predict(X_train_rfe)
print("Linear Support Vector Machines (SVM)")
print("Accuracy on the training set:")
print("Accuracy score:", accuracy_score(y_train, y_pred))
print("Number of correct predictions:", accuracy_score(y_train, y_pred, normalize=False))

Linear Support Vector Machines (SVM)
Accuracy on the training set:
Accuracy score: 1.0
Number of correct predictions: 1722


In [50]:
y_pred=svm.predict(X_val_rfe)
print("Linear Support Vector Machines (SVM)")
print("Accuracy of validation set")
print("Accuracy score:", accuracy_score(y_val, y_pred))
print("Number of correct predictions:", accuracy_score(y_val, y_pred,normalize=False))

Linear Support Vector Machines (SVM)
Accuracy of validation set
Accuracy score: 1.0
Number of correct predictions: 738


In [51]:
y_pred=svm.predict(X_test_rfe)
print("Linear Support Vector Machines (SVM)")
print("Accuracy of testing set")
print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Number of correct predictions:", accuracy_score(y_test, y_pred,normalize=False))

Linear Support Vector Machines (SVM)
Accuracy of testing set
Accuracy score: 1.0
Number of correct predictions: 2460


In [52]:
y_pred=svm.predict(X2_rfe)
print("Linear Support Vector Machines (SVM)")
print("Accuracy of random sample 42")
print("Accuracy score:", accuracy_score(y2, y_pred))
print("Number of correct predictions:", accuracy_score(y2, y_pred,normalize=False))

df_results = pd.DataFrame(y2, y_pred).reset_index()
df_results = df_results.rename(columns={'index':'Actual', 0:'Predicted'})
df_results

Linear Support Vector Machines (SVM)
Accuracy of random sample 42
Accuracy score: 1.0
Number of correct predictions: 1


,Actual,Predicted
0,Fungal infection,Fungal infection
